In [3]:
import folium
import geopandas as gpd

In [4]:
canopy = gpd.read_file('data/canopy.gdb')
property = gpd.read_file('output/property.gpkg')
access_points = gpd.read_file('output/property_accesspoints.gpkg')
reach_75 = gpd.read_file('output/property_reach_75m.gpkg')
reach_100 = gpd.read_file('output/property_reach_100m.gpkg')

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


In [5]:
reach_75 = reach_75.rename(columns = { 'geometry': 'reach_75m'})
reach_100 = reach_100.rename(columns = { 'geometry': 'reach_100m'})
access_points = access_points.rename(columns = {'geometry': 'access_point'})

In [6]:
property = property.merge(
  reach_75,
  how='left',
  on='property_id'
)

property = property.merge(
  reach_100,
  how='left',
  on='property_id'
)

property = property.merge(
  access_points,
  how='left',
  on='property_id'
)

In [7]:
property['reach_75_100m'] = property['reach_100m'].difference(property['reach_75m'])
property['reach_buffer'] = property['reach_75_100m'].buffer(10)
property_geom = property[['property_id', 'geometry', 'access_point', 'reach_75_100m', 'reach_buffer', 'reach_75m']].copy()

# plot

In [8]:
sub_property = property_geom.sample(n=42, random_state=10)

canopy_wgs = canopy.to_crs(4326)
property_wgs = sub_property.to_crs(4326)

geom_cols = [c for c in property_geom.columns if c not in ['property_id', 'geometry']]

for col in geom_cols:
  property_wgs[col] = gpd.GeoSeries(property_wgs[col], crs=property_geom.crs).to_crs(4326)

In [9]:
m = folium.Map(
  location = [property_wgs.geometry.y.mean(), property_wgs.geometry.x.mean()],
  zoom_start = 14,
  tiles='CartoDB positron'
)

for idx, row in property_wgs.iterrows():
  
  prop = row['geometry']
  access = row['access_point']
  reach = row['reach_75_100m']
  reach_75 = row['reach_75m']
  buffer = row['reach_buffer']
  
  folium.CircleMarker(
    [prop.y, prop.x],
    radius=7,
    color='red',
    fill=True,
    fill_opacity=1,
    popup=f"Property {idx}"
  ).add_to(m)
  
  folium.CircleMarker(
    [access.y, access.x],
    radius=4,
    color="blue",
    fill=True,
    fill_opacity=0.9,
    popup=f"Access {idx}"
  ).add_to(m)
  
  if prop.distance(access) > 0:
    folium.PolyLine(
        locations=[
            [prop.y, prop.x],
            [access.y, access.x]
        ],
        color="purple",
        weight=2,
        opacity=0.8,
        dash_array="5,5",
        tooltip=f"Property → Access ({idx})"
    ).add_to(m)
    
  # folium.GeoJson(
  #     reach,
  #     style_function=lambda x: {
  #         "color": "red",
  #         "weight": 3,
  #         "opacity": 0.6
  #     },
  #     tooltip=f"Reachable streets 75-100m ({idx})"
  # ).add_to(m)
  
  folium.GeoJson(
      reach_75,
      style_function=lambda x: {
          "color": "red",
          "weight": 3,
          "opacity": 0.6
      },
      tooltip=f"Reachable streets 75-100m ({idx})"
  ).add_to(m)
  
  # folium.GeoJson(
  #     buffer,
  #     style_function=lambda x: {
  #         "color": "orange",
  #         "fillColor": "orange",
  #         "weight": 1,
  #         "fillOpacity": 0.25
  #     },
  #     tooltip=f"street buffer ({idx})"
  # ).add_to(m)
  
  # possible = canopy_wgs[canopy_wgs.intersects(buffer)]

  # if not possible.empty:
  #     canopy_inside = possible.geometry.intersection(buffer)

  #     canopy_inside = canopy_inside[~canopy_inside.is_empty]

  #     if not canopy_inside.empty:
  #         folium.GeoJson(
  #             canopy_inside,
  #             style_function=lambda x: {
  #                 "color": "darkgreen",
  #                 "fillColor": "darkgreen",
  #                 "weight": 1,
  #                 "fillOpacity": 0.6
  #             }
  #         ).add_to(m)

m